# Feature 6 : Response Speed & Silent Streaks

## Objective

This notebook analyzes participant response behavior and inactivity patterns.

### Input
- parsed_chat.pkl

### Output
- Average Response Time
- Longest Silent Streak

### Export
- response_analysis.pkl

In [1]:
import pickle

In [2]:
DATA_PATH = "../Data/"

with open(DATA_PATH + "parsed_chat.pkl", "rb") as file:
    chat_data = pickle.load(file)

print(f"Loaded {len(chat_data):,} messages successfully.")

Loaded 78,088 messages successfully.


In [3]:
def calculate_response_times(chat_data):
    """
    Calculate all response intervals for every participant.

    Parameters:
        chat_data (list): Parsed chat messages.

    Returns:
        dict: Dictionary containing response intervals in minutes.
    """
    response_times = {}

    for message in chat_data:
        sender = message["sender"]
        if sender not in response_times:
            response_times[sender] = []

    total_messages = len(chat_data)

    for i in range(total_messages - 1):
        current_sender = chat_data[i]["sender"]
        current_time = chat_data[i]["timestamp"]

        for j in range(i + 1, total_messages):
            next_sender = chat_data[j]["sender"]

            if next_sender != current_sender:
                next_time = chat_data[j]["timestamp"]
                time_difference = (next_time - current_time).total_seconds() / 60

                response_times[next_sender].append(time_difference)
                break

    return response_times

In [4]:
response_times = calculate_response_times(chat_data)

print(f"Total Participants : {len(response_times)}")

print("\nSample Participants\n")

count = 0

for participant in response_times:

    print(participant)
    print(response_times[participant][:10])
    print("-" * 60)

    count += 1

    if count == 5:
        break

Total Participants : 81

Sample Participants

Anshley CSE A
[2.0, 45.0, 7.0, 7.0, 5.0, 1.0, 0.0, 0.0, 0.0, 0.0]
------------------------------------------------------------
Kaninika CSE
[13.0, 13.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0]
------------------------------------------------------------
+91 72788 80557
[2.0, 14.0, 27.0, 1.0, 1.0, 1.0, 3.0, 0.0, 0.0, 0.0]
------------------------------------------------------------
Soumili Ghosh CSE
[12.0, 12.0, 12.0, 11.0, 11.0, 0.0, 0.0, 0.0, 0.0, 0.0]
------------------------------------------------------------
Shubham Biswas
[75.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]
------------------------------------------------------------


In [5]:
def calculate_average_response_time(response_times):
    """
    Calculate the average response time for every participant.

    Parameters:
        response_times (dict): Dictionary containing response intervals.

    Returns:
        dict: Dictionary containing average response time in minutes.
    """
    average_response_time = {}

    for participant in response_times:
        intervals = response_times[participant]

        if len(intervals) == 0:
            average_response_time[participant] = 0
        else:
            average_response_time[participant] = sum(intervals) / len(intervals)

    return average_response_time

In [6]:
average_response_time = calculate_average_response_time(response_times)

print("=" * 70)
print("AVERAGE RESPONSE TIME")
print("=" * 70)

count = 0

for participant in average_response_time:
    print(f"{participant:<35}{average_response_time[participant]:>10.2f} min")

    count += 1

    if count == 10:
        break

AVERAGE RESPONSE TIME
Anshley CSE A                           26.75 min
Kaninika CSE                            17.03 min
+91 72788 80557                         17.24 min
Soumili Ghosh CSE                       36.71 min
Shubham Biswas                          28.30 min
Sayan Sheel CSE                         25.07 min
Shambaditya Sarkar CSE                  24.25 min
+91 73842 52736                          4.27 min
+91 70017 73199                         30.16 min
Amit CSE                                15.82 min


In [7]:
from datetime import timedelta

def calculate_silent_streaks(chat_data):
    """
    Calculate the longest silent streak for every participant.

    Parameters:
        chat_data (list): Parsed chat messages.

    Returns:
        dict: Dictionary containing longest silent streak details.
    """
    participant_dates = {}

    for message in chat_data:
        sender = message["sender"]
        message_date = message["timestamp"].date()

        if sender not in participant_dates:
            participant_dates[sender] = set()

        participant_dates[sender].add(message_date)

    start_date = chat_data[0]["timestamp"].date()
    end_date = chat_data[-1]["timestamp"].date()

    silent_streaks = {}

    for participant in participant_dates:

        longest_streak = 0
        current_streak = 0

        longest_start = None
        longest_end = None

        current_start = None

        current_date = start_date

        while current_date <= end_date:

            if current_date in participant_dates[participant]:

                if current_streak > longest_streak:
                    longest_streak = current_streak
                    longest_start = current_start
                    longest_end = current_date - timedelta(days=1)

                current_streak = 0
                current_start = None

            else:

                if current_streak == 0:
                    current_start = current_date

                current_streak += 1

            current_date += timedelta(days=1)

        if current_streak > longest_streak:
            longest_streak = current_streak
            longest_start = current_start
            longest_end = end_date

        silent_streaks[participant] = {
            "days": longest_streak,
            "start_date": longest_start,
            "end_date": longest_end
        }

    return silent_streaks

In [8]:
silent_streaks = calculate_silent_streaks(chat_data)

print("=" * 70)
print("LONGEST SILENT STREAKS")
print("=" * 70)

count = 0

for participant in silent_streaks:
    streak = silent_streaks[participant]

    print(f"Participant : {participant}")
    print(f"Days        : {streak['days']}")
    print(f"Start Date  : {streak['start_date']}")
    print(f"End Date    : {streak['end_date']}")
    print("-" * 70)

    count += 1

    if count == 5:
        break

LONGEST SILENT STREAKS
Participant : Anshley CSE A
Days        : 18
Start Date  : 2024-12-14
End Date    : 2024-12-31
----------------------------------------------------------------------
Participant : Kaninika CSE
Days        : 23
Start Date  : 2026-06-06
End Date    : 2026-06-28
----------------------------------------------------------------------
Participant : +91 72788 80557
Days        : 70
Start Date  : 2025-06-02
End Date    : 2025-08-10
----------------------------------------------------------------------
Participant : Soumili Ghosh CSE
Days        : 24
Start Date  : 2024-11-15
End Date    : 2024-12-08
----------------------------------------------------------------------
Participant : Shubham Biswas
Days        : 51
Start Date  : 2024-11-11
End Date    : 2024-12-31
----------------------------------------------------------------------


In [9]:
feature6_summary = {
    "average_response_time": average_response_time,
    "silent_streaks": silent_streaks
}

In [10]:
print("=" * 70)
print("VALIDATION")
print("=" * 70)

assert len(average_response_time) == len(silent_streaks)
assert len(average_response_time) > 0
assert len(silent_streaks) > 0

print("Feature 6 validation passed.")

VALIDATION
Feature 6 validation passed.


In [11]:
with open(DATA_PATH + "response_analysis.pkl", "wb") as file:
    pickle.dump(feature6_summary, file)

print("response_analysis.pkl exported successfully.")

response_analysis.pkl exported successfully.
